# Episode 2 — `phasor~`: Phase and Basic Waveforms

⏱ About 25 minutes · Difficulty ★☆☆

**Musical element this episode controls: timbre.** Pitch and duration stay the same.

**The equivalent Max patch:** `phasor~ 110`, followed by different objects that reshape the ramp.

**What you will hear:** several timbres at the same pitch (A2) — sine, saw, square, triangle, then a pulse with adjustable "width", and a brightening effect made by "bending the phase".

**By the end you can:**
- explain what phase is and why it equals `phasor~`
- build four basic waveforms from one phase
- change pulse width, from hollow to thin and nasal
- say what aliasing is and when it shows up

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

`play` and `scope` come from Episode 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

freq = 110
dur = 2.0
t = np.arange(int(sr * dur)) / sr

## 1. A new operation: `%`

`%` is read "mod" and means **what is left over after dividing**.

- `7 % 3` → 7 holds two 3s, leaving 1
- `2.75 % 1` → 2.75 holds two 1s, leaving 0.75

So **`% 1` means "keep only the fractional part"**. The Max object is `%~` (or `wrap~ 0 1`).

Run the cell. Note the last line: `% 1` also works on a whole array at once.

In [ ]:
print("7 % 3     =", 7 % 3)
print("2.75 % 1  =", 2.75 % 1)

counting_up = np.array([0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25])
print("before    =", counting_up)
print("after % 1 =", counting_up % 1)

Look at the last two lines: the upper one keeps climbing; the lower one climbs almost to 1, jumps back to 0 and starts over.

## 2. Phase

In Episode 1, `freq * t` was **"how many cycles have elapsed so far"**, and it keeps growing.

`(freq * t) % 1` keeps only the fractional part and answers a different question: **"how far along are we inside the current cycle"**.

- 0 = this cycle just started
- 0.5 = halfway
- jumping back to 0 just before 1 = the next cycle begins

This is called **phase**. It is exactly the 0→1 ramp that `phasor~` outputs.

In [ ]:
cycles = freq * t          # cycles elapsed, keeps growing
phase = cycles % 1         # fractional part only: where we are inside the current cycle

n = int(sr * 0.03)
fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
axes[0].plot(t[:n], cycles[:n])
axes[0].set_ylabel("freq * t")
axes[0].set_title("cycles elapsed (keeps growing)")
axes[1].plot(t[:n], phase[:n])
axes[1].set_ylabel("phase")
axes[1].set_title("phase = (freq * t) % 1   (same as phasor~)")
axes[1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

The upper plot climbs to 3.3 (3.3 cycles in 0.03 seconds). The lower plot is the same thing, reset to zero each time it reaches 1.

### The speed of phase = pitch

`phasor~ 110` runs from 0 to 1 a total of 110 times per second. `phasor~ 220` does it 220 times. The steeper the ramp, the higher the pitch.

**Guess before running:** how many ramps does a 220 Hz phase show in the same 0.03 seconds?

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(t[:n], phase[:n], label="110 Hz")
plt.plot(t[:n], ((220 * t) % 1)[:n], label="220 Hz", alpha=0.7)
plt.xlabel("time (s)")
plt.ylabel("phase")
plt.legend()
plt.show()

Answer: 6.6, exactly twice as many as at 110 Hz.

**Can you listen to phase directly?** Yes; it is a waveform itself (a repeating ramp). But it sits between 0 and 1 rather than centred on 0, so hold on: waveform 2 below takes care of that.

## 3. Turning phase into waveforms

The core idea:

- **phase sets the pitch** (how many runs per second)
- **the shape you turn phase into sets the timbre**

All four waveforms below use **the same `phase`**.

### 3.1 sine

Max: `phasor~` → right inlet (phase inlet) of `cycle~`.

**You should hear:** the same clean hum as in Episode 1.

In [ ]:
x_sine = np.sin(2 * np.pi * phase)

scope(x_sine, "sine")
play(x_sine)

### 3.2 saw

Phase already is a ramp. Just stretch 0→1 to -1→1:

- `2 * phase` → 0 to 2
- `- 1` → -1 to 1

Max: `phasor~` → `*~ 2` → `-~ 1`.

**You should hear:** the same pitch, bright and buzzy.

In [ ]:
x_saw = 2 * phase - 1      # 0..1 → 0..2 → -1..1

scope(x_saw, "saw")
play(x_saw)

### 3.3 square

The rule: output 1 during the first half of the cycle (phase below 0.5), -1 during the second half.

`np.where(condition, A, B)`: put A where the condition holds, B where it does not. Still the whole array at once.

Max: `phasor~` → `<~ 0.5`, then scale to ±1.

**You should hear:** a hollow tone, like an old game console.

In [ ]:
# 1.0 where phase < 0.5, -1.0 everywhere else
x_square = np.where(phase < 0.5, 1.0, -1.0)

scope(x_square, "square")
play(x_square)

### 3.4 triangle

In three steps:

- `phase - 0.5` → a ramp from -0.5 to 0.5
- `np.abs(...)` → absolute value; the negative half flips up into a "V" (0.5 → 0 → 0.5)
- `4 * ... - 1` → stretch to -1 ... 1

Max: `phasor~` → `-~ 0.5` → `abs~` → `*~ 4` → `-~ 1`.

**You should hear:** soft, only slightly brighter than the sine.

In [ ]:
# phase - 0.5: ramp -0.5..0.5 → abs: a V, 0.5..0..0.5 → ×4 then -1: stretched to 1..-1..1
x_tri = 4 * np.abs(phase - 0.5) - 1

scope(x_tri, "triangle")
play(x_tri)

### All four together

From dark to bright: sine → triangle → square → saw. **The sharper the corners and the more jumps in a waveform, the brighter it sounds.**

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 6), sharex=True)
for ax, sig, name in zip(axes, [x_sine, x_tri, x_square, x_saw],
                         ["sine", "triangle", "square", "saw"]):
    ax.plot(t[:n], sig[:n])
    ax.set_ylabel(name)
axes[-1].set_xlabel("time (s)")
plt.show()

## 4. Pulse width

The square's rule was "output 1 while phase is below **0.5**". Replace 0.5 with another number and you change how much of each cycle is "high". That number is the **pulse width**.

- `width = 0.5` → square, hollow
- `width = 0.25` → thinner
- `width = 0.05` → very thin and nasal

**You should hear:** three tones at the same pitch, going from hollow to thin and reedy.

In [ ]:
def pulse(phase, width):
    # width = the fraction of each cycle that is "high"
    return np.where(phase < width, 1.0, -1.0)

for width in [0.5, 0.25, 0.05]:
    scope(pulse(phase, width), f"pulse, width = {width}")
    display(play(pulse(phase, width)))

The first loop has appeared: `for width in [0.5, 0.25, 0.05]:` means "let `width` be each of these three numbers in turn, running the indented lines each time". In Max you would copy the patch three times; here the same code runs three times.

## 5. Bending the phase

So far the flow was: phase → shape.

You can add a step in between: **reshape the phase itself, then hand it to the sine.** This is phase distortion (the method of the Casio CZ synths).

`phase ** 3` is phase cubed. 0 stays 0 and 1 stays 1, but the middle is pushed down: most of the cycle moves slowly and the last stretch rushes.

**You should hear:** three tones — a plain sine, then two that get brighter and more "biting". The pitch does not change.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for power in [1, 2, 4]:
    bent = phase ** power          # 0 and 1 stay put, the middle sags: slow at first, rushing at the end
    axes[0].plot(t[:n], bent[:n], label=f"phase ** {power}")
    axes[1].plot(t[:n], np.sin(2 * np.pi * bent)[:n], label=f"power {power}")
axes[0].set_title("bent phase")
axes[1].set_title("sine of bent phase")
axes[0].legend()
plt.show()

for power in [1, 2, 4]:
    display(play(np.sin(2 * np.pi * phase ** power)))

## 6. Two phase tricks

### One octave up

`(phase * 2) % 1`: multiply phase by 2 (0→2), then `% 1` (two 0→1 ramps). Two runs per cycle → one octave higher.

**You should hear:** the original saw, then a saw an octave higher.

In [ ]:
phase_up = (phase * 2) % 1     # two 0→1 runs per cycle = one octave up

display(play(2 * phase - 1))
display(play(2 * phase_up - 1))

### Phase offset

`(phase + 0.25) % 1`: start the whole cycle a quarter earlier.

**You should hear:** two tones that sound **exactly the same**. The phase offset of a single oscillator is inaudible; it only matters once it is mixed with other sounds.

**You should see:** the second curve has the same shape, shifted left by a quarter cycle.

In [ ]:
phase_shift = (phase + 0.25) % 1     # a quarter cycle early; % 1 wraps whatever passes 1

plt.figure(figsize=(9, 2.5))
plt.plot(t[:n], np.sin(2 * np.pi * phase)[:n], label="offset 0")
plt.plot(t[:n], np.sin(2 * np.pi * phase_shift)[:n], label="offset 0.25")
plt.legend()
plt.xlabel("time (s)")
plt.show()

display(play(np.sin(2 * np.pi * phase)))
display(play(np.sin(2 * np.pi * phase_shift)))

## 7. Aliasing: a problem with these waveforms

A saw and a square contain "instant jumps". A jump implies very high frequencies — in theory infinitely high.

But the sample rate is 44100, which can represent nothing above 22050 Hz. Frequencies beyond that do not vanish; they **fold back** as tones unrelated to the pitch. This is aliasing.

At low pitches it is hard to hear. At high pitches it is obvious.

**You should hear:**
1. a 2500 Hz sine — a clean high tone
2. a 2500 Hz saw — the high tone plus some dissonant, lower junk tones underneath

(The level is reduced; high frequencies are harsh.)

In [ ]:
phase_hi = (2500 * t) % 1

display(play(np.sin(2 * np.pi * phase_hi), gain=0.1))
display(play(2 * phase_hi - 1, gain=0.1))

Using a raw `phasor~` as a saw in Max has exactly the same problem; `saw~` and `rect~` exist to solve it (they are band-limited).

Next episode's wavetable solves it another way: the table only contains a limited number of harmonics.

## Common mistakes

| Symptom | Cause |
|---|---|
| Playing `phase` directly sounds lopsided / thumps | Phase sits between 0 and 1, not centred on 0. Use `2 * phase - 1` |
| `np.where` raises an error | It needs all three: the condition, the value if true, the value if false |
| Forgot `% 1` | `np.sin` does not care (it wraps by itself), but saw and square break completely |

## Exercises

**1.** Make a pulse of width 0.1 at 220 Hz.

<details><summary>Answer</summary>

```python
p220 = (220 * t) % 1
play(pulse(p220, 0.1))
```
</details>

**2.** Make a "half-wave sine": keep the sine where it is above 0, and output 0 where it is below. Hint: `np.maximum(a, 0)` replaces every negative number in `a` with 0. **Guess first:** brighter or darker than a sine?

<details><summary>Answer</summary>

```python
half = np.maximum(np.sin(2 * np.pi * phase), 0)
scope(half, "half-wave sine")
play(half)
```
Brighter. The waveform gained corners.
</details>

**3.** Use the trick from section 6 to make a saw **two** octaves higher.

<details><summary>Answer</summary>

```python
play(2 * ((phase * 4) % 1) - 1)
```
One octave up is times 2, two octaves up is times 4.
</details>

## Recap

| Max | Python |
|---|---|
| `phasor~ 110` | `(110 * t) % 1` |
| `%~ 1` / `wrap~ 0 1` | `% 1` |
| `<~ 0.5` | `phase < 0.5` |
| `selector~` / `?` | `np.where(cond, a, b)` |
| `abs~` | `np.abs(...)` |
| `pow~` | `**` |
| copying a patch three times | `for ... in [...]:` |

- pitch comes from how fast phase runs
- timbre comes from the "phase → shape" step
- waveforms with jumps alias at high pitches

**Next episode:** no more formulas for the shape; store it in a table and read it back — the wavetable.